In [1]:
from openai import OpenAI
from typing import Literal
from pathlib import Path
from pydantic import BaseModel
from dotenv import load_dotenv
import json
import os

Definido Modelo LLM Local

In [ ]:
load_dotenv(dotenv_path=Path(".env"))

MODEL = "sabiazinho-4"

client = OpenAI(
    base_url="https://chat.maritaca.ai/api",
    api_key=os.getenv("MARITACA_API_KEY"),
)

Classe Agent Router

In [3]:
class RouteDecision(BaseModel):
    route: Literal["cag", "rag", "mongo", "agentic", "direct"]
    reason: str
    query: str

In [4]:
def route_question(question: str) -> RouteDecision:
    prompt = f"""
Voce e um router de uma arquitetura RAG/CAG/MCP.

Sua tarefa NAO e responder a pergunta.
Sua tarefa e escolher a melhor rota.

Rotas disponiveis:
- cag: conhecimento cacheado, regras gerais, politicas, FAQ e resumos.
- rag: documentos longos, PDFs, contratos, manuais e trechos especificos.
- mongo: dados estruturados como clientes, pedidos, produtos, usuarios e registros.
- agentic: perguntas que precisam combinar mais de uma fonte ou varios passos.
- direct: conversa simples sem necessidade de base externa.

Pergunta do usuario:
{question}

Responda APENAS JSON valido, sem markdown, neste formato:
{{
  "route": "cag",
  "reason": "motivo curto",
  "query": "consulta otimizada"
}}
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )

    raw_text = response.choices[0].message.content
    json_start = raw_text.find("{")
    json_end = raw_text.rfind("}")

    if json_start == -1 or json_end == -1:
        raise ValueError(f"Resposta sem JSON valido: {raw_text}")

    data = json.loads(raw_text[json_start:json_end + 1])

    return RouteDecision(**data)

In [5]:
def run_cag(query: str) -> str:
    cache_path = Path("cache.json")
    cache = json.loads(cache_path.read_text(encoding="utf-8"))

    prompt = f"""
Voce e um assistente CAG.

Responda usando APENAS o cache abaixo.
Se a informacao nao estiver no cache, diga que nao encontrou no cache.
Responda em portugues do Brasil.

Cache:
{json.dumps(cache, ensure_ascii=False, indent=2)}

Pergunta:
{query}
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )

    return response.choices[0].message.content

In [6]:
def direct_answer(question: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": question}
        ],
    )

    return response.choices[0].message.content

In [7]:
def run_rag(query: str) -> str:
    return "RAG ainda nao implementado."


def run_mongo(query: str) -> str:
    return "Mongo ainda nao implementado."


def agentic_loop(question: str) -> str:
    return "Agentic loop ainda nao implementado."

In [8]:
def answer_question(question: str) -> str:
    decision = route_question(question)

    print(f"Rota escolhida: {decision.route}")
    print(f"Motivo: {decision.reason}")
    print(f"Query: {decision.query}")

    if decision.route == "cag":
        return run_cag(decision.query)

    elif decision.route == "rag":
        return run_rag(decision.query)

    elif decision.route == "mongo":
        return run_mongo(decision.query)

    elif decision.route == "agentic":
        return agentic_loop(question)

    elif decision.route == "direct":
        return direct_answer(question)

    return "Nao consegui escolher uma rota valida."

In [9]:
if __name__ == "__main__":
    question = input("Pergunta: ")

    answer = answer_question(question)

    print("\nResposta:")
    print(answer)

InternalServerError: Error code: 500 - {'error': {'message': 'llama-server reported out-of-memory during startup: ggml_backend_cpu_buffer_type_alloc_buffer: failed to allocate buffer of size 1399947264\nalloc_tensor_range: failed to allocate CUDA_Host buffer of size 1399947264\nerror loading model: unable to allocate CUDA_Host buffer', 'type': 'api_error', 'param': None, 'code': None}}